In [1]:
import os
import json
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# JURECA workaround for matplotlib font cache, got some permission errors from it
os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER','user')}"

USER          = os.environ.get('USER', 'sigursteinsson1')
FINAL_DIR     = Path(f"/p/project1/training2600/{USER}/Final_Project")
PROCESSED_DIR = FINAL_DIR / "processed_data"
RESULTS_DIR   = FINAL_DIR / "results"

# Read class names dynamically from the train folder
surviving_class_names = sorted([
    d.name for d in (PROCESSED_DIR / "train").iterdir() if d.is_dir()
])
NUM_CLASSES = len(surviving_class_names)

plt.rcParams.update({
    "figure.dpi":   100,
    "savefig.dpi":  300,
    "savefig.bbox": "tight",
})

print(f"Classes ({NUM_CLASSES}):")
for i, name in enumerate(surviving_class_names):
    print(f"  {i}: {name}")

Classes (8):
  0: broad_leaved_forest
  1: coniferous_forest
  2: mixed_forest
  3: moors_and_heathland
  4: peat_bogs
  5: transitional_woodland_shrub
  6: water_bodies
  7: water_courses


In [4]:
# Count patches per class per split by reading the folder structure
split_counts = {split: {} for split in ("train", "val", "test")}
for split in ("train", "val", "test"):
    for cls_name in surviving_class_names:
        cls_dir = PROCESSED_DIR / split / cls_name
        n = len(list(cls_dir.glob("*.npy"))) if cls_dir.exists() else 0
        split_counts[split][cls_name] = n

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(NUM_CLASSES)
width = 0.27

train_vals = [split_counts["train"][n] for n in surviving_class_names]
val_vals   = [split_counts["val"][n]   for n in surviving_class_names]
test_vals  = [split_counts["test"][n]  for n in surviving_class_names]

ax.bar(x - width, train_vals, width, label="Train")
ax.bar(x,         val_vals,   width, label="Val")
ax.bar(x + width, test_vals,  width, label="Test")

ax.set_xticks(x)
ax.set_xticklabels(
    [n.replace("_", " ") for n in surviving_class_names],
    rotation=30, ha="right", fontsize=9,
)
ax.set_ylabel("Number of patches")
ax.set_title("Class distribution across train / val / test splits")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, axis="y")

out_path = RESULTS_DIR / "fig1_class_distribution.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")

Saved: /p/project1/training2600/sigursteinsson1/Final_Project/results/fig1_class_distribution.png


In [6]:
# NOTE: our test set is very imbalanced, raw counts would allow coniferous_forest to dominate
# Load saved Prithvi predictions and the precomputed confusion matrix from the previous cell
# This way we can use a precomputed confusion matrix rather than recomputing from preds/labels
preds  = np.load(RESULTS_DIR / "prithvi_test_preds.npy")
labels = np.load(RESULTS_DIR / "prithvi_test_labels.npy")
cm     = np.load(RESULTS_DIR / "prithvi_confusion_matrix.npy")



# Row-normalise: each row sums to 1 (or 0 if no samples for that class)
# We do this since our test set is so imbalanced, we divide each row by its row sum
# so the values represent: "of all the samples that truly belong to class X, what fraction did the model predict as class Y"
# the diagonal of the confusion matrix then has a clear message, it's the recall for each class
# each row sums to 1, allowing for easy comparison


row_sums = cm.sum(axis=1, keepdims=True)
cm_norm  = np.where(row_sums > 0, cm / row_sums, 0.0) # fallback in case a class has 0 true samples, since the divison of it would yield NaN

# Display labels: replace underscores with spaces, no special abbreviation
display_labels = [n.replace("_", " ") for n in surviving_class_names]

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={"label": "Proportion of true class"},
    xticklabels=display_labels,
    yticklabels=display_labels,
    annot_kws={"size": 9},
    linewidths=0.3,
    linecolor="#cccccc",
    vmin=0, vmax=1, # keeps the figure realistic and comparable with the colorbar legend
    ax=ax,
)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Prithvi-EO-2.0 Row-normalised confusion matrix (test set)")
plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
plt.setp(ax.get_yticklabels(), rotation=0)

out_path = RESULTS_DIR / "fig2_confusion_matrix.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")

# Print the diagonal (per-class recall) for reference
print(f"\nDiagonal (per-class recall):")
for i, name in enumerate(surviving_class_names):
    print(f"  {name:<35}  {cm_norm[i, i]:.3f}")

Saved: /p/project1/training2600/sigursteinsson1/Final_Project/results/fig2_confusion_matrix.png

Diagonal (per-class recall):
  broad_leaved_forest                  0.099
  coniferous_forest                    0.443
  mixed_forest                         0.064
  moors_and_heathland                  0.036
  peat_bogs                            0.334
  transitional_woodland_shrub          0.093
  water_bodies                         0.035
  water_courses                        0.022


In [21]:
# Build an index of (file_path, true_label) for the test set in the same order
# the DataLoader iterated through them.
# CorineDataset in Notebook 03 iterates classes alphabetically and then
test_samples = []
for cls_idx, cls_name in enumerate(surviving_class_names):
    cls_dir = PROCESSED_DIR / "test" / cls_name
    if not cls_dir.exists():
        continue
    for fpath in cls_dir.glob("*.npy"):
        test_samples.append((fpath, cls_idx))

# Sanity check — the labels we recover from the folder must match
# the labels saved during inference.
assert len(test_samples) == len(labels), (
    f"Mismatch: {len(test_samples)} files in folder vs {len(labels)} saved labels."
)
folder_labels = np.array([s[1] for s in test_samples])
assert np.array_equal(folder_labels, labels), (
    "Folder-derived label order does not match saved labels "
    "DataLoader iteration order may have changed."
)

# For each class: pick one correctly classified and one misclassified example
examples = []   # list of (file_path, true_idx, pred_idx, is_correct)
rng = np.random.default_rng(seed=42)

for cls_idx in range(NUM_CLASSES):
    mask = (labels == cls_idx)
    indices = np.where(mask)[0]
    correct_indices   = indices[preds[indices] == cls_idx]
    incorrect_indices = indices[preds[indices] != cls_idx]

    if len(correct_indices) > 0:
        idx = rng.choice(correct_indices)
        examples.append((test_samples[idx][0], cls_idx, int(preds[idx]), True))
    if len(incorrect_indices) > 0:
        idx = rng.choice(incorrect_indices)
        examples.append((test_samples[idx][0], cls_idx, int(preds[idx]), False))

n_examples = len(examples)
n_cols = 4
n_rows = int(np.ceil(n_examples / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.0, n_rows * 2.2))
axes = np.atleast_2d(axes)

for i, (fpath, true_idx, pred_idx, correct) in enumerate(examples):
    ax = axes[i // n_cols, i % n_cols]
    patch = np.load(fpath).astype(np.float32)  # shape (16, 3, 3)

    # Channel order: [B02_Mar, B03_Mar, B04_Mar, B08_Mar, B02_May, ..., B08_Oct]
    # May acquisition (greenup) RGB: B04_May=6, B03_May=5, B02_May=4
    rgb = np.stack([patch[6], patch[5], patch[4]], axis=-1)
    rgb = np.clip(rgb, 0, 1)

    ax.imshow(rgb, interpolation="nearest")
    ax.set_xticks([]); ax.set_yticks([])

    true_name = surviving_class_names[true_idx].replace("_", " ")
    pred_name = surviving_class_names[pred_idx].replace("_", " ")
    title_color = "#1a7f1a" if correct else "#b22222"
    ax.set_title(
        f"True: {true_name}\nPred: {pred_name}",
        fontsize=7, color=title_color, wrap=True,
    )
    for spine in ax.spines.values():
        spine.set_edgecolor(title_color)
        spine.set_linewidth(1.5)

# Hide unused subplots
for j in range(n_examples, n_rows * n_cols):
    axes[j // n_cols, j % n_cols].axis("off")

fig.suptitle(
    "Sample 3×3 test patches (RGB from May acquisition), "
    "green border: correct prediction, red border: misclassified",
    fontsize=10, y=1.02,
)
plt.subplots_adjust(hspace=0.6, wspace=0.3, top=0.94)

out_path = RESULTS_DIR / "fig3_qualitative_patches.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")

Saved: /p/project1/training2600/sigursteinsson1/Final_Project/results/fig3_qualitative_patches.png


In [23]:
with open(RESULTS_DIR / "all_results.json") as f:
    all_results = json.load(f)

# Display order: naive baselines, balanced baselines then Prithvi
model_display_order = [
    ("rf_naive",          "RF\n(naive)"),
    ("mlp_naive",         "MLP\n(naive)"),
    ("rf_balanced",       "RF\n(balanced)"),
    ("mlp_balanced",      "MLP\n(balanced)"),
    ("prithvi_eo_v2_300", "Prithvi-EO-2.0\n(fine-tuned)"),
]

def get_metric(entry, key_options):
    """Pull a metric value, handling both naming conventions used in the JSON."""
    for k in key_options:
        if k in entry:
            return entry[k]
    return None

labels_x = [d[1] for d in model_display_order]
oa_vals  = [get_metric(all_results[k], ["test_oa", "overall_accuracy"]) for k, _ in model_display_order]
f1_vals  = [get_metric(all_results[k], ["test_macro_f1", "macro_f1"])   for k, _ in model_display_order]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(labels_x))
width = 0.38

bars_oa = ax.bar(x - width/2, oa_vals, width, label="Test OA (Is the model right?)")
bars_f1 = ax.bar(x + width/2, f1_vals, width, label="Test Macro F1 (Is the model actually useful?)")

# Numeric labels above each bar
for bars in (bars_oa, bars_f1):
    for b in bars:
        h = b.get_height()
        ax.annotate(
            f"{h:.3f}",
            xy=(b.get_x() + b.get_width() / 2, h),
            xytext=(0, 3), textcoords="offset points",
            ha="center", fontsize=8,
        )

ax.set_xticks(x)
ax.set_xticklabels(labels_x, fontsize=9)
ax.set_ylabel("Score (test set)")
ax.set_ylim(0, max(max(oa_vals), max(f1_vals)) * 1.18)
ax.set_title("Test set performance across all five models")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3, axis="y")

out_path = RESULTS_DIR / "fig4_model_comparison.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")

Saved: /p/project1/training2600/sigursteinsson1/Final_Project/results/fig4_model_comparison.png


In [24]:
with open(RESULTS_DIR / "prithvi_training_history.json") as f:
    history = json.load(f)

epochs       = [h["epoch"]        for h in history]
train_losses = [h["train_loss"]   for h in history]
val_losses   = [h["val_loss"]     for h in history]
val_oas      = [h["val_oa"]       for h in history]
val_f1s      = [h["val_macro_f1"] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Loss curves
ax1.plot(epochs, train_losses, marker="o", markersize=4, label="Train loss")
ax1.plot(epochs, val_losses,   marker="s", markersize=4, label="Val loss")
ax1.axvline(x=5.5, color="#888888", linestyle="--", alpha=0.7)
ax1.text(5.6, ax1.get_ylim()[1] * 0.95, "Backbone\nunfrozen",
         fontsize=8, color="#555555", va="top")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Cross-entropy loss")
ax1.set_title("(a) Training and validation loss")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

# Validation metrics
ax2.plot(epochs, val_oas, marker="o", markersize=4, label="Val OA")
ax2.plot(epochs, val_f1s, marker="s", markersize=4, label="Val Macro F1")
ax2.axvline(x=5.5, color="#888888", linestyle="--", alpha=0.7)
ax2.text(5.6, ax2.get_ylim()[1] * 0.95, "Backbone\nunfrozen",
         fontsize=8, color="#555555", va="top")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_title("(b) Validation metrics")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()

out_path = RESULTS_DIR / "fig5_training_curves.png"
plt.savefig(out_path)
plt.show()
print(f"Saved: {out_path}")

Saved: /p/project1/training2600/sigursteinsson1/Final_Project/results/fig5_training_curves.png
